# 🚀 Startup Success Prediction — Enhanced Edition
### Hyperparameter Tuning + NLP Features + Time-Series Features

---

## 🗺️ What's New in This Version

This notebook extends the baseline with three powerful improvements:

| Enhancement | What it adds |
|-------------|--------------|
| **GridSearchCV Tuning** | Automatically finds the best model hyperparameters instead of using defaults |
| **NLP Features** | Extracts signals from startup descriptions using TF-IDF text analysis |
| **Time-Series Features** | Engineers temporal signals from founding year, funding dates, and company age |

---

## 📦 Step 1: Install & Import Dependencies

### What is this doing?

We install everything from the baseline, plus a few new libraries needed for the enhancements:
- `scipy` — used internally by TF-IDF for sparse matrix operations
- `sklearn.feature_extraction.text` — provides `TfidfVectorizer` for NLP features
- `sklearn.decomposition` — provides `TruncatedSVD` to compress TF-IDF from thousands of columns down to a manageable size
- `sklearn.pipeline` — lets us chain preprocessing + model into a single object for cleaner GridSearchCV
- `sklearn.model_selection.GridSearchCV` — exhaustively tries all combinations of hyperparameters and picks the best one

In [ ]:
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn plotly scipy -q

In [ ]:
import os, glob, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.labelsize': 12})

print('✅ All libraries imported successfully!')

---
## 🔑 Step 2: Download Dataset via kagglehub

### What is this doing?

Same as before — `kagglehub` downloads the Crunchbase VC Investments dataset in one line.
On first run it prompts for your Kaggle username and API key (get it from https://www.kaggle.com/settings → API → **Create New Token**).

In [ ]:
import kagglehub

path = kagglehub.dataset_download('arindam235/startup-investments-crunchbase')
print('✅ Dataset downloaded to:', path)

csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
print('Found:', csv_files)

---
## 📂 Step 3: Load & Inspect Data

In [ ]:
df = pd.read_csv(csv_files[0], encoding='latin-1', low_memory=False)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Columns:', df.columns.tolist())
df.head()

---
## 🧹 Step 4: Data Cleaning & Target Engineering

### What is this doing?

Same cleaning as before, but now we also **preserve raw date and text columns** before dropping anything — we'll need them for NLP and time-series feature engineering in later steps.

- `status` → binary `success` (1 = acquired/IPO, 0 = everything else)
- Drop columns > 60% missing AFTER extracting what we need from them

In [ ]:
# ── Save raw text & date columns before any dropping ─────────────────────────
# NLP: look for a description column
desc_candidates = ['description', 'short_description', 'overview', 'company_description']
desc_col = next((c for c in desc_candidates if c in df.columns), None)
print(f'Description column found: {desc_col}')

# Time: look for date columns
date_candidates = ['founded_at', 'founded_year', 'first_funding_at', 'last_funding_at']
date_cols = [c for c in date_candidates if c in df.columns]
print(f'Date columns found: {date_cols}')

# Preserve these before dropping high-NaN columns
raw_text = df[desc_col].copy() if desc_col else None
raw_dates = df[date_cols].copy() if date_cols else pd.DataFrame()

# ── Define target ─────────────────────────────────────────────────────────────
df['success'] = df['status'].apply(
    lambda x: 1 if str(x).lower() in ['acquired', 'ipo'] else 0
) if 'status' in df.columns else None

print('\nTarget distribution:')
print(df['success'].value_counts())

# ── Drop high-missing columns ─────────────────────────────────────────────────
df_clean = df.loc[:, df.isnull().mean() < 0.60].copy()
print(f'\nColumns after cleaning: {df_clean.shape[1]}')

---
## ⏳ Step 5: Time-Series Feature Engineering

### What is this doing?

Raw dates (like `founded_at = '2008-03-15'`) are useless to a model as-is. We **engineer meaningful numeric signals** from them:

| New Feature | What it captures |
|-------------|------------------|
| `founding_year` | The era the startup was born in (dot-com era vs mobile era vs AI era) |
| `company_age_years` | How long the company existed before its outcome (older = more proven) |
| `days_to_first_funding` | How quickly did it attract investors after founding? (fast = hot deal) |
| `funding_duration_days` | How long was the active fundraising window? (longer = multiple rounds over time) |
| `years_since_last_funding` | How recently was the last injection of capital? (recent = still active) |
| `founded_in_recession` | Was it founded during 2008–2010 recession? (tests resilience signal) |
| `founding_decade` | Categorical decade bin (1990s, 2000s, 2010s, 2020s) |

**Why do these matter?**  
A startup founded in 2005 during the Web 2.0 boom and still operating in 2015 has a very different profile from one founded in 2020. Time gives the model crucial context that raw funding numbers alone can't capture.

In [ ]:
REFERENCE_YEAR = 2015  # approximate mid-point of dataset

ts_features = pd.DataFrame(index=df_clean.index)

# ── Parse founding year / date ────────────────────────────────────────────────
if 'founded_at' in raw_dates.columns:
    founded = pd.to_datetime(raw_dates['founded_at'], errors='coerce')
    ts_features['founding_year']  = founded.dt.year
    ts_features['founding_month'] = founded.dt.month   # seasonality signal
    ts_features['company_age_years'] = REFERENCE_YEAR - founded.dt.year
    ts_features['founded_in_recession'] = founded.dt.year.between(2008, 2010).astype(int)
    ts_features['founding_decade'] = (founded.dt.year // 10 * 10)  # e.g. 2000, 2010
elif 'founded_year' in raw_dates.columns:
    fy = pd.to_numeric(raw_dates['founded_year'], errors='coerce')
    ts_features['founding_year']  = fy
    ts_features['company_age_years'] = REFERENCE_YEAR - fy
    ts_features['founded_in_recession'] = fy.between(2008, 2010).astype(int)
    ts_features['founding_decade'] = (fy // 10 * 10)

# ── Parse first / last funding dates ─────────────────────────────────────────
if 'first_funding_at' in raw_dates.columns and 'founded_at' in raw_dates.columns:
    first_fund = pd.to_datetime(raw_dates['first_funding_at'], errors='coerce')
    ts_features['days_to_first_funding'] = (first_fund - founded).dt.days

if 'first_funding_at' in raw_dates.columns and 'last_funding_at' in raw_dates.columns:
    first_fund = pd.to_datetime(raw_dates['first_funding_at'], errors='coerce')
    last_fund  = pd.to_datetime(raw_dates['last_funding_at'],  errors='coerce')
    ts_features['funding_duration_days']   = (last_fund - first_fund).dt.days
    ts_features['years_since_last_funding'] = REFERENCE_YEAR - last_fund.dt.year

print(f'Time-series features created: {ts_features.shape[1]}')
print(ts_features.describe())

In [ ]:
# ── Visualize time-series features ───────────────────────────────────────────
ts_plot = ts_features.copy()
ts_plot['success'] = df_clean['success'].values

time_plot_cols = [c for c in ['founding_year', 'company_age_years',
                               'days_to_first_funding', 'funding_duration_days']
                  if c in ts_plot.columns]

if time_plot_cols:
    fig, axes = plt.subplots(1, len(time_plot_cols), figsize=(5 * len(time_plot_cols), 4))
    if len(time_plot_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, time_plot_cols):
        for outcome, color, label in zip([0, 1], ['#E07B54', '#4CAF81'],
                                          ['Not Successful', 'Successful']):
            data = ts_plot[ts_plot['success'] == outcome][col].dropna()
            ax.hist(data, bins=30, alpha=0.55, color=color, density=True, label=label)
        ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
        ax.legend(fontsize=8)

    plt.suptitle('Time-Series Features by Outcome', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No time-series features available to plot (dataset may lack date columns).')

In [ ]:
# ── Success rate by founding decade ──────────────────────────────────────────
if 'founding_decade' in ts_features.columns:
    decade_df = pd.DataFrame({
        'founding_decade': ts_features['founding_decade'].values,
        'success': df_clean['success'].values
    }).dropna()

    decade_success = (
        decade_df.groupby('founding_decade')['success']
        .agg(['mean', 'count'])
        .reset_index()
    )
    decade_success.columns = ['decade', 'success_rate', 'count']
    decade_success = decade_success[decade_success['count'] >= 10]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(decade_success['decade'].astype(int).astype(str),
                  decade_success['success_rate'] * 100,
                  color=sns.color_palette('crest', len(decade_success)),
                  edgecolor='white', width=5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('Founding Decade')
    ax.set_ylabel('Success Rate (%)')
    ax.set_title('Startup Success Rate by Founding Decade', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 📝 Step 6: NLP Feature Engineering (TF-IDF)

### What is this doing?

Startup descriptions contain rich signals that pure numeric data misses. A description mentioning *"machine learning"*, *"FDA approved"*, or *"enterprise SaaS"* tells you far more about the company's prospects than funding numbers alone.

We convert text into numbers using **TF-IDF** (Term Frequency–Inverse Document Frequency):

**TF-IDF explained:**
- **Term Frequency (TF)**: How often does a word appear in *this* description?
- **Inverse Document Frequency (IDF)**: How rare is this word *across all* descriptions?
- **Result**: Common words like "the", "and", "company" get low scores. Rare, meaningful words like "genomics" or "blockchain" get high scores.

**The problem:** TF-IDF produces thousands of columns (one per unique word). Too many for a model to handle efficiently.

**The solution — TruncatedSVD (Latent Semantic Analysis):**  
Compresses thousands of TF-IDF columns into just **50 "topic" dimensions** that capture the core meaning. Think of it like taking thousands of specific words and distilling them into 50 thematic clusters (e.g. "healthcare", "fintech", "consumer apps").

**Preprocessing the text:**
- Lowercase everything
- Remove punctuation
- `stop_words='english'` removes common words ("the", "is", "and") that carry no signal
- `ngram_range=(1,2)` captures both single words (`"machine"`) and pairs (`"machine learning"`)

In [ ]:
import re

N_SVD_COMPONENTS = 50  # number of latent "topic" dimensions to compress TF-IDF into

nlp_features = pd.DataFrame(index=df_clean.index)

if raw_text is not None:
    print(f'Description column: "{desc_col}"')
    print(f'Non-null descriptions: {raw_text.notna().sum():,} / {len(raw_text):,}')

    # ── Clean text ────────────────────────────────────────────────────────────
    def clean_text(text):
        if pd.isna(text):
            return ''
        text = str(text).lower()
        text = re.sub(r'[^a-z\s]', ' ', text)   # remove non-alpha characters
        text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
        return text

    cleaned_text = raw_text.apply(clean_text)
    print('\nSample cleaned descriptions:')
    print(cleaned_text[cleaned_text != ''].head(3).values)

    # ── TF-IDF Vectorization ──────────────────────────────────────────────────
    # max_features=3000: keep only the 3000 most informative words
    # ngram_range=(1,2): include both single words and 2-word phrases
    # min_df=3: ignore words that appear in fewer than 3 descriptions
    tfidf = TfidfVectorizer(
        max_features=3000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3,
        sublinear_tf=True     # apply log(1+TF) smoothing
    )
    X_tfidf = tfidf.fit_transform(cleaned_text)
    print(f'\nTF-IDF matrix shape: {X_tfidf.shape}  (rows x unique n-grams)')

    # ── Compress with TruncatedSVD (LSA) ─────────────────────────────────────
    # Reduces 3000 columns → 50 semantic topic dimensions
    svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=42)
    X_svd = svd.fit_transform(X_tfidf)

    explained_var = svd.explained_variance_ratio_.sum()
    print(f'SVD explained variance with {N_SVD_COMPONENTS} components: {explained_var:.1%}')

    # Store as named columns
    svd_cols = [f'nlp_topic_{i}' for i in range(N_SVD_COMPONENTS)]
    nlp_features = pd.DataFrame(X_svd, columns=svd_cols, index=df_clean.index)
    print(f'NLP features created: {nlp_features.shape[1]} columns')

else:
    print('No description column found. NLP features will be skipped.')
    print('Available columns:', df.columns.tolist())

In [ ]:
# ── Visualize top TF-IDF words for successful vs not successful startups ──────
if raw_text is not None:
    success_mask = df_clean['success'] == 1
    fail_mask    = df_clean['success'] == 0

    # Re-fit simple TF-IDF per group for word-level analysis
    tfidf_vis = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1,1))
    X_all = tfidf_vis.fit_transform(cleaned_text)
    vocab = np.array(tfidf_vis.get_feature_names_out())

    mean_success = np.asarray(X_all[success_mask.values].mean(axis=0)).flatten()
    mean_fail    = np.asarray(X_all[fail_mask.values].mean(axis=0)).flatten()

    top_n = 15
    top_success_idx = mean_success.argsort()[::-1][:top_n]
    top_fail_idx    = mean_fail.argsort()[::-1][:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].barh(vocab[top_success_idx][::-1], mean_success[top_success_idx][::-1],
                 color='#4CAF81', edgecolor='white')
    axes[0].set_title('Top Words in SUCCESSFUL Startup Descriptions', fontweight='bold')
    axes[0].set_xlabel('Mean TF-IDF Score')

    axes[1].barh(vocab[top_fail_idx][::-1], mean_fail[top_fail_idx][::-1],
                 color='#E07B54', edgecolor='white')
    axes[1].set_title('Top Words in NOT SUCCESSFUL Startup Descriptions', fontweight='bold')
    axes[1].set_xlabel('Mean TF-IDF Score')

    plt.tight_layout()
    plt.show()

---
## 🔧 Step 7: Combine All Features

### What is this doing?

Now we bring together all three feature groups into one unified feature matrix:

1. **Baseline features** — funding amounts, country, market, investor types (from original columns)
2. **Time-series features** — founding year, company age, days to first funding, etc.
3. **NLP features** — 50 semantic topic dimensions extracted from descriptions

We use `pd.concat(axis=1)` to merge them column-by-column. If a feature group is empty (e.g. no description column exists in the dataset), it's simply skipped — so the notebook works on any version of the dataset.

After merging, we apply the same preprocessing pipeline:
- Label encode any remaining text columns
- Impute missing values with median
- 80/20 stratified train/test split
- SMOTE to handle class imbalance
- StandardScaler for Logistic Regression

In [ ]:
# ── Baseline feature selection ────────────────────────────────────────────────
feature_candidates = [
    'funding_rounds', 'funding_total_usd', 'country_code', 'market',
    'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC',
    'has_roundD', 'avg_participants', 'is_top500'
]
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
base_feats = [c for c in feature_candidates if c in df_clean.columns]
extra_num  = [c for c in num_cols if c not in base_feats + ['success']]
base_feats = list(dict.fromkeys(base_feats + extra_num))

base_df = df_clean[base_feats].copy()

# Fix funding column formatting
if 'funding_total_usd' in base_df.columns:
    base_df['funding_total_usd'] = pd.to_numeric(
        base_df['funding_total_usd'].astype(str).str.replace(',', ''), errors='coerce'
    )

# ── Combine all feature groups ────────────────────────────────────────────────
frames = [base_df]
if not ts_features.empty:
    frames.append(ts_features)
if not nlp_features.empty:
    frames.append(nlp_features)

X_all = pd.concat(frames, axis=1)
y     = df_clean['success']

print(f'Total features: {X_all.shape[1]}')
print(f'  Baseline:    {len(base_feats)}')
print(f'  Time-series: {ts_features.shape[1]}')
print(f'  NLP topics:  {nlp_features.shape[1]}')

In [ ]:
# ── Encode, impute, split, SMOTE, scale ──────────────────────────────────────
le = LabelEncoder()
for col in X_all.select_dtypes(include='object').columns:
    X_all[col] = le.fit_transform(X_all[col].astype(str))

imp = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imp.fit_transform(X_all), columns=X_all.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {dict(zip(*np.unique(y_res, return_counts=True)))}')

scaler = StandardScaler()
X_res_sc = scaler.fit_transform(X_res)
X_test_sc = scaler.transform(X_test)

---
## 🔍 Step 8: Hyperparameter Tuning with GridSearchCV

### What is this doing?

Every ML model has **hyperparameters** — settings that control how the model learns (not learned from data, but set by the developer). In the baseline, we used default values. Here we **automatically search** for better values.

**GridSearchCV explained:**
- You provide a **grid** of possible values for each hyperparameter
- It tries **every combination** (the "grid")
- For each combination, it runs **5-fold cross-validation** (splits training data into 5 parts, trains on 4, validates on 1, repeats 5 times)
- It picks the combination with the best average validation score

**Hyperparameters we tune per model:**

| Model | Hyperparameters | What they control |
|-------|----------------|-------------------|
| Logistic Regression | `C` | Regularization strength (smaller C = simpler model) |
| Random Forest | `n_estimators`, `max_depth`, `min_samples_split` | Number of trees, how deep they grow, minimum samples to split |
| XGBoost | `n_estimators`, `max_depth`, `learning_rate`, `subsample` | Depth, step size, random row sampling per tree |

> ⚠️ **Note:** GridSearch can be slow. We use a focused grid with few options per parameter. For larger grids, use `RandomizedSearchCV` instead — it samples random combinations rather than exhaustive search.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ─────────────────────────────────────────────────────────────────────────────
# 8a: Logistic Regression
# C = inverse regularization strength: smaller C = stronger regularization = simpler model
# solver='saga' is the fastest solver for large datasets
# ─────────────────────────────────────────────────────────────────────────────
print('🔍 Tuning Logistic Regression...')
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='saga', random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0
)
lr_grid.fit(X_res_sc, y_res)
print(f'  Best C={lr_grid.best_params_["C"]}  →  CV AUC={lr_grid.best_score_:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8b: Random Forest
# n_estimators: number of trees (more = better but slower)
# max_depth: how deep each tree can grow (None = unlimited, prone to overfit)
# min_samples_split: min samples required to split a node (higher = simpler trees)
# ─────────────────────────────────────────────────────────────────────────────
print('🔍 Tuning Random Forest...')
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth':    [None, 10, 20],
        'min_samples_split': [2, 5]
    },
    cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0
)
rf_grid.fit(X_res, y_res)
print(f'  Best params: {rf_grid.best_params_}  →  CV AUC={rf_grid.best_score_:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8c: XGBoost
# learning_rate (eta): step size shrinkage — smaller = more conservative updates
# subsample: fraction of rows to use per tree (prevents overfitting)
# colsample_bytree: fraction of features to use per tree
# max_depth: max depth per tree
# ─────────────────────────────────────────────────────────────────────────────
print('🔍 Tuning XGBoost...')
xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', verbosity=0, random_state=42),
    param_grid={
        'n_estimators':  [100, 200],
        'max_depth':     [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample':     [0.8, 1.0]
    },
    cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0
)
xgb_grid.fit(X_res, y_res)
print(f'  Best params: {xgb_grid.best_params_}  →  CV AUC={xgb_grid.best_score_:.4f}')

In [ ]:
# ── GridSearch results comparison chart ──────────────────────────────────────
gs_results = {
    'Logistic Regression': lr_grid,
    'Random Forest':       rf_grid,
    'XGBoost':             xgb_grid
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, gs) in zip(axes, gs_results.items()):
    results_df = pd.DataFrame(gs.cv_results_)
    sorted_df  = results_df.sort_values('mean_test_score', ascending=False).head(10)

    # Create a compact label from params
    param_cols = [c for c in results_df.columns if c.startswith('param_')]
    sorted_df['label'] = sorted_df[param_cols].apply(
        lambda row: '\n'.join([f"{c.replace('param_','')[:8]}={v}" for c, v in row.items()]), axis=1
    )

    colors = ['#4CAF81' if i == 0 else '#5b8cdb' for i in range(len(sorted_df))]
    bars = ax.barh(range(len(sorted_df)),
                   sorted_df['mean_test_score'].values[::-1],
                   xerr=sorted_df['std_test_score'].values[::-1],
                   color=colors[::-1], edgecolor='white', capsize=4)
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(
        [f"Combo {i+1}" for i in range(len(sorted_df)-1, -1, -1)],
        fontsize=8
    )
    ax.set_xlabel('CV ROC-AUC')
    ax.set_title(f'{name}\nBest AUC={gs.best_score_:.4f} (green)', fontweight='bold', fontsize=11)
    ax.set_xlim(sorted_df['mean_test_score'].min() - 0.02, 1.0)

plt.suptitle('GridSearchCV Results — Top 10 Hyperparameter Combinations per Model',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🤖 Step 9: Evaluate Tuned Models

### What is this doing?

We now take the **best estimator** from each GridSearchCV (the model with optimal hyperparameters) and evaluate it on the **held-out test set** it has never seen.

`gs.best_estimator_` is the model already retrained on the full training set using the best-found hyperparameters — no need to retrain manually.

In [ ]:
tuned_results = {}

for name, gs in gs_results.items():
    best = gs.best_estimator_

    if 'Logistic' in name:
        preds = best.predict(X_test_sc)
        proba = best.predict_proba(X_test_sc)[:, 1]
    else:
        preds = best.predict(X_test)
        proba = best.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    tuned_results[name] = {'model': best, 'preds': preds,
                            'proba': proba, 'accuracy': acc, 'auc': auc}
    print(f'{name:25s}  →  Accuracy: {acc:.4f}  |  ROC-AUC: {auc:.4f}')

In [ ]:
# ── ROC Curves for tuned models ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
palette = ['#3B82F6', '#10B981', '#F59E0B']

for (name, res), color in zip(tuned_results.items(), palette):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})",
            linewidth=2.5, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Tuned Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, res) in zip(axes, tuned_results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    ConfusionMatrixDisplay(cm, display_labels=['Not Successful', 'Successful']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices — Tuned Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance for best tuned model ───────────────────────────────────
best_tuned_name = max(tuned_results, key=lambda k: tuned_results[k]['auc'])
best_tuned_model = tuned_results[best_tuned_name]['model']
print(f'Best tuned model: {best_tuned_name}  (AUC={tuned_results[best_tuned_name]["auc"]:.4f})')

if hasattr(best_tuned_model, 'feature_importances_'):
    fi = pd.Series(best_tuned_model.feature_importances_, index=X_test.columns)

    # Separate feature groups for color coding
    def feat_group(name):
        if name.startswith('nlp_topic'): return 'NLP'
        if name in ts_features.columns:  return 'Time-Series'
        return 'Baseline'

    fi_df = fi.sort_values(ascending=False).head(25).reset_index()
    fi_df.columns = ['feature', 'importance']
    fi_df['group'] = fi_df['feature'].apply(feat_group)

    group_colors = {'Baseline': '#5b8cdb', 'Time-Series': '#F59E0B', 'NLP': '#10B981'}
    colors = fi_df['group'].map(group_colors)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1],
            color=colors[::-1].values, edgecolor='white')
    ax.set_title(f'Top 25 Feature Importances — {best_tuned_name}\n'
                  '(🔵 Baseline  🟡 Time-Series  🟢 NLP)',
                  fontweight='bold')
    ax.set_xlabel('Importance Score')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.show()

---
## 📊 Step 10: Before vs After Comparison

### What is this doing?

We compare the **tuned models with enhanced features** against a simple baseline (default XGBoost, no tuning, no extra features) to quantify how much each improvement contributed.

This is the most important chart for communicating the value of the work done — it shows the **ROC-AUC gain** from:
1. Adding time-series features
2. Adding NLP features  
3. Tuning hyperparameters with GridSearchCV

In [ ]:
# ── Baseline (default XGBoost, no extra features) ─────────────────────────────
# Re-train baseline on just the original numeric features for fair comparison
base_only = df_clean[base_feats].copy()
if 'funding_total_usd' in base_only.columns:
    base_only['funding_total_usd'] = pd.to_numeric(
        base_only['funding_total_usd'].astype(str).str.replace(',', ''), errors='coerce'
    )
for col in base_only.select_dtypes(include='object').columns:
    base_only[col] = LabelEncoder().fit_transform(base_only[col].astype(str))

X_b_imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(base_only),
                        columns=base_only.columns)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X_b_imp, y, test_size=0.2, random_state=42, stratify=y
)
Xb_res, yb_res = SMOTE(random_state=42).fit_resample(Xb_tr, yb_tr)
baseline_xgb = XGBClassifier(n_estimators=150, random_state=42,
                               eval_metric='logloss', verbosity=0)
baseline_xgb.fit(Xb_res, yb_res)
baseline_auc = roc_auc_score(yb_te, baseline_xgb.predict_proba(Xb_te)[:, 1])
print(f'Baseline XGBoost AUC (default params, no extra features): {baseline_auc:.4f}')

# ── Summary comparison ────────────────────────────────────────────────────────
comparison = [
    {'Model': 'Baseline XGBoost (default)', 'ROC-AUC': baseline_auc, 'Type': 'Baseline'},
] + [
    {'Model': f'{n} (tuned + all features)', 'ROC-AUC': r['auc'], 'Type': 'Enhanced'}
    for n, r in tuned_results.items()
]
comp_df = pd.DataFrame(comparison).sort_values('ROC-AUC', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#E07B54' if t == 'Baseline' else '#4CAF81' for t in comp_df['Type']]
bars = ax.barh(comp_df['Model'], comp_df['ROC-AUC'], color=colors, edgecolor='white')
for bar, val in zip(bars, comp_df['ROC-AUC']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
ax.axvline(baseline_auc, linestyle='--', color='#E07B54', alpha=0.5, linewidth=1.5,
           label=f'Baseline: {baseline_auc:.4f}')
ax.set_xlabel('ROC-AUC Score')
ax.set_title('Baseline vs Enhanced Models — ROC-AUC Comparison', fontweight='bold', fontsize=14)
ax.set_xlim(comp_df['ROC-AUC'].min() - 0.02, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Classification report for best tuned model ────────────────────────────────
print(f'\n📊 Classification Report — {best_tuned_name} (Tuned + All Features)\n')
print(classification_report(
    y_test, tuned_results[best_tuned_name]['preds'],
    target_names=['Not Successful', 'Successful']
))

---
## 🔮 Step 11: Predict on a New Startup

### What is this doing?

Same as before, but now the prediction uses the **best tuned model with all enhanced features**.

The input now includes time-series features (founding year, company age) in addition to the original numeric features. NLP features default to zeros when no description is provided — meaning the model uses only structured data for startups without a description.

> 💡 **Try customizing:** Change `founding_year` to see if newer startups score differently, or increase `funding_rounds` and `funding_total_usd` to simulate a well-funded startup.

In [ ]:
# Build a new startup input — all zeros by default, then override what you know
new_startup = pd.DataFrame([{col: 0 for col in X_test.columns}])

# ── Baseline features ─────────────────────────────────────────────────────────
if 'funding_rounds'    in new_startup.columns: new_startup['funding_rounds']    = 3
if 'funding_total_usd' in new_startup.columns: new_startup['funding_total_usd'] = 5_000_000
if 'has_VC'            in new_startup.columns: new_startup['has_VC']            = 1
if 'has_roundA'        in new_startup.columns: new_startup['has_roundA']        = 1
if 'has_roundB'        in new_startup.columns: new_startup['has_roundB']        = 1

# ── Time-series features ──────────────────────────────────────────────────────
if 'founding_year'       in new_startup.columns: new_startup['founding_year']       = 2012
if 'company_age_years'   in new_startup.columns: new_startup['company_age_years']   = 3
if 'founding_decade'     in new_startup.columns: new_startup['founding_decade']     = 2010
if 'days_to_first_funding' in new_startup.columns: new_startup['days_to_first_funding'] = 180
if 'funding_duration_days' in new_startup.columns: new_startup['funding_duration_days'] = 730

# NLP features remain zero (no description provided — model uses structured data only)

# ── Run prediction ────────────────────────────────────────────────────────────
new_imp = pd.DataFrame(imp.transform(new_startup), columns=X_test.columns)

best_model = tuned_results[best_tuned_name]['model']
if 'Logistic' in best_tuned_name:
    prob = best_model.predict_proba(scaler.transform(new_imp))[0][1]
else:
    prob = best_model.predict_proba(new_imp)[0][1]

print(f'\n🚀 Predicted Success Probability: {prob:.2%}')
print(f'   Verdict: {"✅ Likely Successful" if prob >= 0.5 else "❌ Likely Not Successful"}')

# Confidence gauge
fig, ax = plt.subplots(figsize=(8, 1.5))
ax.barh([''], [1], color='#E07B54', height=0.5)
ax.barh([''], [prob], color='#4CAF81', height=0.5)
ax.axvline(0.5, color='white', linewidth=2, linestyle='--')
ax.text(prob, 0, f' {prob:.1%}', va='center', fontsize=14, fontweight='bold', color='white')
ax.set_xlim(0, 1)
ax.set_title('Success Probability Gauge', fontweight='bold')
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticklabels(['0%', '25%', '50%\n(threshold)', '75%', '100%'])
ax.yaxis.set_visible(False)
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Enhancement | Technique | What it did |
|-------------|-----------|-------------|
| **GridSearchCV** | 5-fold CV over param grids for LR, RF, XGBoost | Found optimal hyperparameters automatically |
| **NLP Features** | TF-IDF (3000 n-grams) → TruncatedSVD (50 topics) | Extracted semantic signals from startup descriptions |
| **Time-Series Features** | Parsed founding/funding dates into 7 numeric signals | Added temporal context: company age, funding speed, era |
| **Feature Importance** | Color-coded by feature group | Reveals *which* features drive predictions most |
| **Before/After Chart** | Baseline vs tuned+enhanced | Quantifies the improvement from each addition |

---

### 💡 Further Improvements

- **RandomizedSearchCV** — Sample random hyperparameter combinations instead of exhaustive grid (much faster for large grids)
- **Stacking / Ensembling** — Use the 3 tuned models as inputs to a meta-learner for further accuracy gains
- **SHAP values** — Explain individual predictions (why did *this* startup get predicted as successful?)
- **Richer NLP** — Replace TF-IDF with sentence embeddings (BERT, sentence-transformers) for deeper semantic understanding
- **More time features** — Market conditions at founding time (interest rates, VC investment volume that year)